# CauST Quickstart Tutorial

This notebook demonstrates the **CauST** (Causal Gene Intervention for Robust Spatial Domain Identification) pipeline end-to-end.

## Overview
1. Create synthetic spatial transcriptomics data
2. Run single-slice CauST pipeline
3. Inspect causal gene scores
4. Visualize spatial domains
5. Run multi-slice mode with LODO validation
6. Save and reload models

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import anndata as ad
from scipy.sparse import csr_matrix
import scanpy as sc
import matplotlib.pyplot as plt

from caust import CauST
from caust.evaluate.metrics import evaluate_single_slice
from caust.visualize.plots import plot_spatial_domains, plot_causal_scores

## 1. Create Synthetic Data

We create a toy AnnData with 3 spatially separated clusters and distinct expression profiles.

In [ ]:
def make_synthetic_adata(n_obs=300, n_vars=200, n_clusters=3, seed=42):
    rng = np.random.default_rng(seed)
    obs_per = n_obs // n_clusters
    
    # Expression: each cluster has a distinct mean
    X_parts = []
    for i in range(n_clusters):
        base = rng.poisson(lam=5 + i * 3, size=(obs_per, n_vars)).astype(np.float32)
        # Make some genes cluster-specific (causal)
        causal_start = i * 20
        base[:, causal_start:causal_start+20] += 10 * (i + 1)
        X_parts.append(base)
    X = np.vstack(X_parts)
    
    adata = ad.AnnData(X=csr_matrix(X))
    adata.obs_names = [f'spot_{i}' for i in range(X.shape[0])]
    adata.var_names = [f'gene_{j}' for j in range(n_vars)]
    
    # Spatial coordinates: 3 separated clusters
    coords = np.vstack([
        rng.normal([0, 0], 2, size=(obs_per, 2)),
        rng.normal([20, 0], 2, size=(obs_per, 2)),
        rng.normal([10, 17], 2, size=(obs_per, 2)),
    ])
    adata.obsm['spatial'] = coords
    
    # Ground-truth labels
    labels = np.repeat(np.arange(n_clusters), obs_per)
    adata.obs['ground_truth'] = labels.astype(str)
    
    return adata

adata = make_synthetic_adata()
print(f'AnnData: {adata.n_obs} spots × {adata.n_vars} genes')
print(f'Spatial coords shape: {adata.obsm["spatial"].shape}')
adata

## 2. Run CauST Single-Slice

The `fit_transform()` method runs the full pipeline:
- Builds spatial graph
- Trains GAT autoencoder
- Computes causal gene scores via perturbation
- Filters/reweights genes
- Clusters latent space with K-Means

In [ ]:
model = CauST(
    n_causal_genes  = 50,
    n_clusters      = 3,
    epochs          = 100,
    scoring_method  = 'gradient',   # fast for demo
    filter_mode     = 'filter_and_reweight',
    verbose         = True,
)

adata_out = model.fit_transform(adata.copy())
print(f'\nOutput: {adata_out.n_obs} spots × {adata_out.n_vars} genes')
print(f'Domain labels: {np.unique(adata_out.obs["caust_domain"].values)}')

## 3. Evaluate Results

In [ ]:
labels_pred = adata_out.obs['caust_domain'].astype(int).values
latent_Z = adata_out.obsm['caust_latent']
labels_true = adata.obs['ground_truth'].values

metrics = evaluate_single_slice(labels_pred, latent_Z, labels_true)
for k, v in metrics.items():
    print(f'  {k}: {v:.4f}')

## 4. Inspect Causal Gene Scores

CauST identifies which genes are most important for spatial domain formation.

In [ ]:
# Top 15 causal genes
top_genes = model.get_top_causal_genes(n=15)
print('Top 15 causal genes:')
for gene, score in top_genes:
    print(f'  {gene}: {score:.4f}')

# Bar chart
genes, scores = zip(*top_genes)
fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(range(len(genes)), scores, color='steelblue')
ax.set_yticks(range(len(genes)))
ax.set_yticklabels(genes)
ax.set_xlabel('Causal Score')
ax.set_title('Top Causal Genes')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 5. Visualize Spatial Domains

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

coords = adata.obsm['spatial']

# Ground truth
ax = axes[0]
gt = adata.obs['ground_truth'].astype(int).values
scatter = ax.scatter(coords[:, 0], coords[:, 1], c=gt, cmap='tab10', s=10)
ax.set_title('Ground Truth')
ax.set_aspect('equal')

# CauST predicted
ax = axes[1]
pred = adata_out.obs['caust_domain'].astype(int).values
# Align obs to original coords
scatter = ax.scatter(coords[:len(pred), 0], coords[:len(pred), 1], c=pred, cmap='tab10', s=10)
ax.set_title('CauST Predicted Domains')
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

## 6. Multi-Slice Mode

CauST supports training across multiple tissue slices with IRM-style invariance.

In [ ]:
# Create 3 synthetic slices (simulating different donors)
slices = {
    'slice_A': make_synthetic_adata(n_obs=150, seed=10),
    'slice_B': make_synthetic_adata(n_obs=150, seed=20),
    'slice_C': make_synthetic_adata(n_obs=150, seed=30),
}

donor_map = {
    'slice_A': 'Donor1',
    'slice_B': 'Donor2',
    'slice_C': 'Donor2',
}

multi_model = CauST(
    n_causal_genes=50, n_clusters=3, epochs=50,
    scoring_method='gradient', verbose=True,
)

results = multi_model.fit_multi_slice(slices, donor_map=donor_map)

print(f'\nResults for {len(results)} slices:')
for sid, adata_r in results.items():
    print(f'  {sid}: {adata_r.n_obs} spots, {adata_r.n_vars} genes')

## 7. Save & Reload Model

In [ ]:
import tempfile, os

with tempfile.TemporaryDirectory() as tmpdir:
    save_path = os.path.join(tmpdir, 'caust_model')
    model.save(save_path)
    print(f'Saved to: {save_path}')
    print(f'Files: {os.listdir(save_path)}')
    
    # Reload without specifying n_genes
    loaded = CauST.load(save_path)
    out2 = loaded.transform(adata.copy())
    print(f'\nReloaded model output: {out2.n_obs} spots')
    print('Reload successful!')

## Next Steps

- Try with real data: run `scripts/01_download_data.py` to get DLPFC, Mouse Brain, etc.
- Run full benchmark: `scripts/05_benchmark.py`
- Explore multi-slice LODO: `scripts/04_run_multi_slice.py`
- Check the [GUIDE.md](../GUIDE.md) for full documentation.